# Lecture 9 · Notebook 3 — An LLM judge: position bias and the order-swapping trick
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture09/lecture09_3_llm_judge_bias.ipynb)

**Companion to [Chapter 2 of the video lecture](https://nlp-course-cuhksz.github.io/lecture09.html#p2-s23)** (HuatuoGPT's reward model) and
**[Chapter 3](https://nlp-course-cuhksz.github.io/lecture09.html#p3-s27)** (pairwise evaluation with GPT-4 and doctors as judges). HuatuoGPT's RL stage uses an AI judge, made more
reliable by two tricks:

* **anchor the judge to a reference**: every judgment is grounded in a real doctor's answer (RLMF, slide 22), and
* **ask twice with the order swapped**, because LLM judges tend to prefer whichever answer comes first; HuatuoGPT keeps a preference
  only if both orders agree (slide 23).

Here you will measure both effects with a deliberately small judge, where they are easy to see. Each test pair is built from a MedQA
question: one response states the correct option, the other a wrong one, so we know the right verdict.

Runs on a free CPU runtime in about 5 minutes.

> ⚠️ **Not medical advice.** This notebook studies how language models *behave* on medical test questions. The model is tiny and often
> wrong; nothing it writes here is a diagnosis or a treatment recommendation. For health questions, ask a qualified clinician.

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, json, random, re, time, collections
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # ~1 GB; try "Qwen/Qwen2.5-1.5B-Instruct" on a T4 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print("device:", device, "| parameters:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M")

In [ ]:
from huggingface_hub import hf_hub_download
path = hf_hub_download("GBaker/MedQA-USMLE-4-options", "phrases_no_exclude_test.jsonl", repo_type="dataset")
rows = [json.loads(l) for l in open(path, encoding="utf-8")]
random.seed(1)
N = 60                                   # each pair costs 4 forward passes (2 orders x with/without reference)
pairs = []
for it in random.sample(rows, N):
    wrong = random.choice([v for k, v in it["options"].items() if k != it["answer_idx"]])
    pairs.append({"q": it["question"], "good": f"The answer is: {it['answer']}.", "bad": f"The answer is: {wrong}.", "ref": it["answer"]})
print(pairs[0]["q"][:300], "...\n  good:", pairs[0]["good"], "\n  bad: ", pairs[0]["bad"])

## 1 · The judge, asked in both orders
The judge sees the question and two responses, labelled A and B, and we read P(A) vs. P(B) from its next token. Every pair is judged
twice: once with the correct response as A, once with it as B. Optionally the judge also sees a **reference answer**, as RLMF gives
ChatGPT the doctor's answer.

In [ ]:
AB = [tok.encode(x, add_special_tokens=False)[0] for x in ["A", "B"]]

def judge_prompt(q, r1, r2, ref=None):
    body = f"Question:\n{q}\n\nResponse A: {r1}\nResponse B: {r2}\n\n"
    if ref is not None:
        body += f"Reference answer written by a doctor: {ref}\n\n"
    body += "Which response is medically correct? Answer with A or B only."
    msgs = [{"role": "system", "content": "You are a strict medical examiner grading two responses."}, {"role": "user", "content": body}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def p_A(q, r1, r2, ref=None):
    x = tok(judge_prompt(q, r1, r2, ref), return_tensors="pt").to(device)
    return float(model(**x).logits[0, -1, AB].float().softmax(-1)[0])   # P(judge says "A")

t0 = time.time()
J = {}
for name, use_ref in (("no reference", False), ("with reference", True)):
    pg = np.array([p_A(p["q"], p["good"], p["bad"], p["ref"] if use_ref else None) for p in pairs])   # good shown as A
    pb = np.array([p_A(p["q"], p["bad"], p["good"], p["ref"] if use_ref else None) for p in pairs])   # bad shown as A
    J[name] = (pg, pb)
print(f"[{time.time()-t0:.0f}s]")
for name, (pg, pb) in J.items():
    print(f"{name:15s} mean P(A): {pg.mean():.2f} when the good answer is A, {pb.mean():.2f} when the bad answer is A")

## 2 · Three ways to turn the judge's output into a verdict

1. **One order, threshold 0.5**: what you get if you ask once. We report accuracy separately for "good shown first" and "good shown second".
2. **Agreement filter** (HuatuoGPT, slide 23): keep a pair only if both orders give the same verdict, and score those.
3. **Average both orders**: the judge prefers the good answer if P(good) averaged over the two orders is above 0.5, i.e.
   `(P(A | good is A) + P(B | good is B)) / 2 > 0.5`. A constant preference for one position cancels out.

In [ ]:
def verdicts(pg, pb):
    ok_first = pg > 0.5                     # good shown as A, judge says A
    ok_second = pb < 0.5                    # good shown as B, judge says B
    agree = ok_first == ok_second
    avg = (pg + (1 - pb)) / 2 > 0.5
    return {"good shown first": ok_first.mean(), "good shown second": ok_second.mean(),
            "pairs kept by agreement": agree.mean(), "accuracy on kept pairs": ok_first[agree].mean() if agree.any() else float("nan"),
            "average of both orders": avg.mean()}

R = {name: verdicts(*v) for name, v in J.items()}
for name, r in R.items():
    print(name); [print(f"   {k:24s} {v:.0%}" if v == v else f"   {k:24s} n/a (no pair kept)") for k, v in r.items()]

keys = list(next(iter(R.values())))
x = np.arange(len(keys))
plt.figure(figsize=(9, 3.4))
for j, (name, color) in enumerate((("no reference", "#999999"), ("with reference", "#4c72b0"))):
    plt.bar(x + (j - 0.5) * 0.4, [0 if R[name][k] != R[name][k] else R[name][k] for k in keys], 0.4, label=name, color=color)
plt.axhline(0.5, ls="--", c="gray"); plt.xticks(x, [k.replace(" ", "\n", 2) for k in keys], fontsize=8); plt.ylim(0, 1.05)
plt.legend(loc="upper right"); plt.title(f"A {MODEL_ID.split('/')[-1]} judge on {N} MedQA answer pairs"); plt.show()

What to look for:

* **Position bias.** The small judge says "A" almost regardless of content, so with one order it looks perfect when the good answer
  happens to be first and useless when it is second. A single-order judgment from this judge tells you only where the answer was placed.
* **Agreement filter.** With a judge this biased, almost no pair gives the same verdict in both orders, so the filter keeps almost nothing.
  That is a useful alarm: a low agreement rate tells you the judge cannot be trusted *before* you train a reward model on it (slide 23).
* **Averaging both orders** cancels the constant preference for "A" and reveals the small signal underneath.
* **Anchoring.** With the doctor's reference, the averaged verdict is right far more often: a hard medical question has become an easier
  *matching* task. That is the idea of RLMF (slide 22): the AI provides scale, the doctor provides correctness. Rubric grading
  (HealthBench, slide 40) pushes the same idea further.

Frontier-size judges are much less biased than this 0.5B model, but the lesson holds at every scale: measure the bias, ask in both
orders, and anchor the judge to expert references whenever you have them.

## Exercises
1. **Length bias.** Make the wrong response longer and more confident (e.g. add *"This is the classic presentation; other options are
   clearly incorrect."*). How does the averaged verdict change, with and without the reference?
2. Replace the reference with a **wrong** reference (another option). Does the judge follow the reference or its own knowledge?
   What does that imply for RLMF when the doctor's answer is mistaken?
3. Make the judge **pointwise**: show one response at a time and ask *"Is the response medically correct? yes/no"*. Compare P(yes) for the
   good and the bad response. Does a pointwise judge avoid position bias? What new bias might it have?
4. Turn the judge into a **rubric grader**: write three criteria for one question (e.g. *names the correct diagnosis*, *does not recommend
   a contraindicated drug*, …), ask yes/no for each, and sum the points, as HealthBench does.

**Back to the lecture:** [{SITE}]({SITE}#map)